# Interactive mining-detector inference (annotated)

This notebook is the interactive twin of `inference_pipeline.py`: the same `InferenceEngine`, driven cell by cell for small experiments and debugging rather than long production runs.

**Pipeline recap.** A region (GeoJSON, EPSG:4326) is tiled into 576x576-pixel Sentinel-2 tiles (10 m/pixel, ~5.76 km per side) on the Descartes Labs DLTile UTM grid. For each tile, Google Earth Engine builds a cloud-masked median composite (`COPERNICUS/S2_HARMONIZED` joined with the Cloud Score+ band `cs_cdf`; pixels masked where `cs_cdf < clear_threshold = 0.6`; per-pixel median over the date window), downloaded via `ee.data.computePixels` as a NumPy array of uint16 digital numbers (reflectance x 10000). Each tile is cut into 48x48-pixel chips (stride 24 -> 529 overlapping chips per tile, ~23 ha each); chips are normalized (/10000, clipped to [0, 1]), bicubic-resized to 224x224, embedded by a frozen SSL4EO ViT-S/16 (384-d output), and classified by a Keras ensemble of six 384->64->16->1 sigmoid MLPs. The mean of the six sigmoids is the chip "confidence"; chips above `pred_threshold` are saved as GeoJSON squares.

**Cell flow.** (0) imports + autoreload magics; (1) region/model/dates and the three config dataclasses; (2) tile the region and construct the engine; (3) single-tile debug prediction; (4) full bulk run over all tiles; (5) interactive map of detections.

**Requirements.** Google Earth Engine auth (importing `gee` calls `ee.Initialize()`; set the `EE_PROJECT` env var) and model weights under `../models/`. In this fork only inference is live -- training code is archived under `_archive/`.

In [ ]:
# ============================================================
# Cell 0 -- Imports and notebook conveniences
# ============================================================
# --- Standard library ---
# datetime: handy for timestamping ad-hoc experiment outputs.
from datetime import datetime
# logging: the InferenceEngine reports progress through a logger we wire up below.
import logging
# os: environment/filesystem access (e.g., gee.py reads the EE_PROJECT env var).
import os
# Path: used later to derive the region label via Path(region_path).stem.
from pathlib import Path
# sys: provides sys.stdout as the logging stream target.
import sys

# --- Third party ---
# geopandas: reads the region GeoJSON; predictions come back as GeoDataFrames.
import geopandas as gpd

# --- Local modules (this notebook lives in gee/, so plain imports resolve) ---
# gee: the config dataclasses (DataConfig / InferenceConfig / MaskConfig) and
# the InferenceEngine (GEE composite download, SSL4EO ViT-S/16 embedding,
# Keras MLP-ensemble classification, optional SAM2 masking).
# SIDE EFFECT: importing gee calls ee.Initialize() at import time -- Google
# Earth Engine auth must be configured and EE_PROJECT set beforehand.
import gee
# tile_utils: DLTile-grid tiling helpers (create_tiles, CenteredTile, ...).
import tile_utils

# IPython magics: auto-reload edited local modules (gee.py, tile_utils.py)
# without restarting the kernel -- convenient while iterating on the code.
%load_ext autoreload
%autoreload 2

## Configuration
Region, model, date window, and the three config dataclasses (`DataConfig` for imagery source and cloud threshold, `InferenceConfig` for model paths and chip geometry, `MaskConfig` for SAM2 -- unused here since `run_sam2=False`).

In [ ]:
# ============================================================
# Cell 1 -- Run parameters and the three config dataclasses
# ============================================================
# Region to scan: a GeoJSON of boundary polygons (EPSG:4326) under
# ../data/boundaries/. Swap the name here to point at another region.
region_name = 'test_region_small'
region_path = f'../data/boundaries/{region_name}.geojson'

# Classifier head: a Keras .h5 ensemble of six 384->64->16->1 sigmoid MLPs
# operating on 384-d SSL4EO embeddings. In this fork, detection GeoJSONs land
# under ../data/outputs/<model_version>/ (48px_v0.X-SSL4EO-MLPensemble).
model_name = '48px_v0.X-SSL4EO-MLPensemble_2025-10-21'
model_path = f'../models/{model_name}.h5'

# Compositing window: every Sentinel-2 scene in [start_date, end_date) feeds
# the per-pixel median composite (one static composite per tile).
start_date='2024-01-01'
end_date='2024-12-31'
    
# DataConfig -- imagery source and cloud masking.
#   tilesize=576: tile edge in pixels; at 10 m/px that is ~5.76 km per side.
#   pad=0: no extra border pixels around each tile.
#   collection="S2L1C": COPERNICUS/S2_HARMONIZED (top-of-atmosphere L1C,
#     13 bands), joined with the Cloud Score+ band cs_cdf for masking.
#   clear_threshold=0.6: pixels with cs_cdf < 0.6 are masked out before the
#     median is taken -- raise it to mask more aggressively.
#   max_workers=4: concurrent GEE tile downloads in bulk_predict's producers.
#   image_cache_dir=None: do not cache downloaded tiles to disk.
data_config = gee.DataConfig(
    tilesize=576,
    pad=0, 
    collection="S2L1C",
    clear_threshold=0.6,
    max_workers=4, # Turn this down from 8 if cacheing, to ease memory pressure 
    image_cache_dir=None,
)

# InferenceConfig -- model paths and chip/prediction geometry.
#   pred_threshold=0.85: keep chips whose confidence (mean of the six MLP
#     sigmoids) exceeds 0.85 -- a precision-leaning operating point.
#   stride_ratio=2: chip stride = geo_chip_size // 2 = 24 px, so adjacent
#     48x48 chips overlap by half -> 529 chips per 576-px tile.
#   geo_chip_size=48: chip edge in pixels (~480 m; ~23 ha at 10 m/px).
#     Chips are normalized (/10000, clipped to [0, 1]) and bicubic-resized to
#     224x224 before the frozen SSL4EO ViT-S/16 embeds them (384-d output).
#   embeddings_cache_dir=None: recompute embeddings; no parquet cache.
#   run_sam2=False: skip SAM2 segmentation, so MaskConfig below stays inert.
inference_config = gee.InferenceConfig(
    model_path=model_path,
    pred_threshold=0.85,
    stride_ratio=2, 
    geo_chip_size=48,
    embeddings_cache_dir=None,
    run_sam2=False
)

# MaskConfig -- SAM2 repo/checkpoint paths and mask-smoothing parameters.
# Constructed only to satisfy the engine's signature; unused while
# run_sam2=False.
mask_config = gee.MaskConfig()


## Tiling and engine construction
Cover the region with DLTile-grid tiles, then build the `InferenceEngine` (loads the classifier ensemble and the SSL4EO backbone; the GEE composite stays lazy).

In [ ]:
# ============================================================
# Cell 2 -- Tile the region and construct the InferenceEngine
# ============================================================
# Read the boundary file and expose its geometry column as a plain
# GeoJSON-like mapping (__geo_interface__ dict, EPSG:4326) -- the input
# format tile_utils.create_tiles expects.
region = gpd.read_file(region_path).geometry.__geo_interface__
# Cover the region with 576x576-px tiles snapped to the Descartes Labs
# DLTile UTM grid at 10 m/px. Returns a list of DLTile objects carrying
# key, crs, bounds, geotrans, and an EPSG:4326 geometry.
tiles = tile_utils.create_tiles(region, data_config.tilesize, data_config.pad)
print(f"Created {len(tiles):,} tiles")

# Send the engine's log records to the notebook output by attaching a
# stdout StreamHandler to the root logger.
logger = logging.getLogger()
stream_handler = logging.StreamHandler(sys.stdout)
logger.addHandler(stream_handler)

# Build the engine. This (a) loads the Keras six-MLP ensemble from
# model_path, (b) loads the frozen SSL4EO ViT-S/16 backbone (PyTorch), and
# (c) defines the lazy GEE cloud-masked median composite for the window.
# Nothing is downloaded yet -- pixels are fetched per tile at predict time.
engine = gee.InferenceEngine(
        start_date=start_date,
        end_date=end_date,
        data_config=data_config,
        config=inference_config,
        mask_config=mask_config,
        logger=logger,
    )

## Single-tile debug
Run the full download / chip / embed / classify chain on one tile and inspect the resulting GeoDataFrame of positive chips.

In [ ]:
# ============================================================
# Cell 3 -- Single-tile debug prediction
# ============================================================
# predict_on_tile runs the full chain on one tile, synchronously:
#   1. download this tile's composite via ee.data.computePixels ->
#      (576, 576, 13) array of uint16 digital numbers (reflectance x 10000);
#   2. cut into 529 overlapping 48x48-px chips (stride 24);
#   3. normalize (/10000, clip to [0, 1]), bicubic-resize to 224x224;
#   4. embed with the frozen SSL4EO ViT-S/16 -> (529, 384) float array;
#   5. classify with the six-MLP ensemble; confidence = mean of 6 sigmoids;
#   6. keep chips with confidence > pred_threshold (0.85).
# Single tile inference for debugging
preds_gdf = engine.predict_on_tile(tiles[0])
# Result: a GeoDataFrame (EPSG:4326) of positive chips -- square geometries
# (48x48 px, ~23 ha each) plus a float "confidence" column. Often empty for
# a background tile.
display(preds_gdf)

## Bulk inference
Producer/consumer run over all tiles; detections are written incrementally to `../data/outputs/<model_version>/`.

In [ ]:
# ============================================================
# Cell 4 -- Bulk inference over every tile in the region
# ============================================================
# Producer/consumer run: producer threads download tile composites
# concurrently (max_workers=4) while a single consumer serializes the GPU
# work (SSL4EO embedding + Keras ensemble). Failed tiles are retried
# (InferenceConfig.tries). Predictions are merged and written INCREMENTALLY
# to ../data/outputs/<model_version>/, so partial results survive an
# interrupted run; Path(region_path).stem labels the output GeoJSON.
# Returns the cumulative GeoDataFrame of all positive chips.
preds = engine.bulk_predict(tiles, Path(region_path).stem)

## Visualize detections
Interactive folium map of the detected chips.

In [ ]:
# ============================================================
# Cell 5 -- Interactive map of detections
# ============================================================
# GeoDataFrame.explore() renders an interactive folium/leaflet map inline in
# the notebook. The filter keeps chips with confidence > 0.85 -- redundant
# while pred_threshold is already 0.85, but useful if you lower the
# threshold above and want to sweep operating points here. Each square is
# one ~23 ha detection chip; overlapping squares mark strong clusters.
preds[preds.confidence > 0.85].explore()